# Stage 3 (PyTorch DL): Full Suite with PyTorch & CUDA Acceleration

This notebook replaces the TensorFlow/Keras deep learning models with **PyTorch equivalents**. It runs the full suite of models on a **10% sample** of each data segment, maintaining memory safety and enabling CUDA acceleration for PyTorch, XGBoost, and LightGBM.

## 1. Setup and Imports

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import gc  # Garbage Collector interface
import sys

# Scikit-Learn models and utilities
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import MinMaxScaler

# Gradient Boosting models
import xgboost as xgb
import lightgbm as lgb

# Classical Time Series models
from prophet import Prophet
from statsmodels.tsa.api import ExponentialSmoothing

try:
    import pmdarima as pm

    PMDARIMA_AVAILABLE = True
    print("pmdarima found. SARIMA model is available.")
except ImportError:
    PMDARIMA_AVAILABLE = False
    print("pmdarima not found. SARIMA model is disabled.")

# Deep Learning models with PyTorch
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader

    PYTORCH_AVAILABLE = True
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"PyTorch found (version {torch.__version__}). DL models are available on device: '{device}'.")
except ImportError:
    PYTORCH_AVAILABLE = False
    print("PyTorch not found. DL models are disabled.")

# --- Configuration ---
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (16, 8)
sns.set_style('whitegrid')

SEGMENTS_DIR = 'data/segments/'

pmdarima not found. SARIMA model is disabled.
PyTorch found (version 2.7.0+cu128). DL models are available on device: 'cuda'.


## 2. Memory Optimization Utility

In [2]:
def reduce_mem_usage(df, verbose=True):
    """Iterate through all the columns of a dataframe and modify the data type to reduce memory usage."""
    start_mem = df.memory_usage().sum() / 1024 ** 2
    if verbose:
        print(f'Memory usage of dataframe is {start_mem:.2f} MB')

    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object and col_type.name != 'category' and 'datetime' not in col_type.name:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)

    end_mem = df.memory_usage().sum() / 1024 ** 2
    if verbose:
        print(f'Memory usage after optimization is: {end_mem:.2f} MB')
        print(f'Decreased by {(start_mem - end_mem) / start_mem * 100:.1f}%')

    return df

## 3. Metric and Evaluation Functions

In [3]:
def calculate_mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100


def calculate_wape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)) * 100


def calculate_smape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    numerator = np.abs(y_pred - y_true)
    denominator = (np.abs(y_true) + np.abs(y_pred))
    return 100 * np.mean(numerator / denominator)


def calculate_metrics(y_true, y_pred, is_solar):
    metrics = {
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred)
    }
    if is_solar:
        metrics['WAPE'] = calculate_wape(y_true, y_pred)
        metrics['sMAPE'] = calculate_smape(y_true, y_pred)
    else:
        metrics['MAPE'] = calculate_mape(y_true, y_pred)
    return metrics

## 4. Feature and Target Preparation

In [4]:
def get_features_and_target(df):
    TARGET_COL = 'BaseLoad'
    cols_to_drop = [
        TARGET_COL, 'TradeDate', 'TradeTime', 'LoadProfile', 'RateGroup', 'Submission',
        'LossAdjustedLoad', 'LoadBL', 'LoadLAL', 'GenBL', 'GenLAL', 'Solar_Status', 'Created'
    ]
    features = [col for col in df.columns if col not in cols_to_drop]
    X = df[features]
    y = df[TARGET_COL]
    return X, y

## 5. Walk-Forward Cross-Validation Pipelines

In [5]:
def run_sklearn_validation(df, model_func, is_solar):
    """Performs walk-forward validation for scikit-learn compatible models."""
    print(f"--- Running validation for: {model_func.__name__} ---")
    N_SPLITS = 5
    TEST_PERIOD_HOURS = 24 * 7
    tscv = TimeSeriesSplit(n_splits=N_SPLITS, test_size=TEST_PERIOD_HOURS)
    X, y = get_features_and_target(df)
    all_metrics = []
    
    if len(df) < (N_SPLITS * TEST_PERIOD_HOURS) + TEST_PERIOD_HOURS:
        print(f"  Skipping: Dataset with {len(df)} rows is too small for validation.")
        return {}
    
    for i, (train_index, test_index) in enumerate(tscv.split(X)):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        print(f"  Fold {i + 1}/{N_SPLITS}: Training on {len(X_train)} samples, testing on {len(X_test)} samples...")
        model = model_func()

        if isinstance(model, xgb.XGBRegressor):
            model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
        elif isinstance(model, lgb.LGBMRegressor):
            model.fit(X_train, y_train, eval_set=[(X_test, y_test)], callbacks=[lgb.early_stopping(10, verbose=False)])
        else:
            model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        all_metrics.append(calculate_metrics(y_test, y_pred, is_solar))
        del X_train, X_test, y_train, y_test, model, y_pred;
        gc.collect()

    return pd.DataFrame(all_metrics).mean().to_dict()

In [6]:
def run_univariate_validation(df, model_name, model_logic, is_solar):
    """Generic validation loop for univariate models like Prophet, SARIMA, ETS."""
    print(f"--- Running validation for: {model_name} ---")
    N_SPLITS = 5
    TEST_PERIOD_HOURS = 24 * 7
    tscv = TimeSeriesSplit(n_splits=N_SPLITS, test_size=TEST_PERIOD_HOURS)
    all_metrics = []

    if len(df) < (N_SPLITS * TEST_PERIOD_HOURS) + TEST_PERIOD_HOURS:
        print(f"  Skipping: Dataset with {len(df)} rows is too small for validation.")
        return {}

    for i, (train_index, test_index) in enumerate(tscv.split(df)):
        df_train, df_test = df.iloc[train_index], df.iloc[test_index]
        print(f"  Fold {i + 1}/{N_SPLITS}: Training on {len(df_train)} samples, testing on {len(df_test)} samples...")
        y_pred = model_logic(df_train, len(df_test))
        y_test = df_test['BaseLoad'].values
        all_metrics.append(calculate_metrics(y_test, y_pred, is_solar))
        del df_train, df_test, y_pred, y_test;
        gc.collect()

    return pd.DataFrame(all_metrics).mean().to_dict()

In [7]:
def run_pytorch_validation(df, model_name, model_class, is_solar):
    """Generic validation loop for PyTorch Deep Learning models."""
    if not PYTORCH_AVAILABLE: return {}
    print(f"--- Running validation for: {model_name} ---")
    N_SPLITS = 5
    TEST_PERIOD_HOURS = 24 * 7
    TIMESTEPS = 24 * 7

    if len(df) < (N_SPLITS * TEST_PERIOD_HOURS) + TEST_PERIOD_HOURS + TIMESTEPS:
        print(f"  Skipping: Dataset with {len(df)} rows is too small for DL validation.")
        return {}

    X, y = get_features_and_target(df)
    tscv = TimeSeriesSplit(n_splits=N_SPLITS, test_size=TEST_PERIOD_HOURS)
    all_metrics = []

    x_scaler, y_scaler = MinMaxScaler(), MinMaxScaler()
    X_scaled = x_scaler.fit_transform(X)
    y_scaled = y_scaler.fit_transform(y.values.reshape(-1, 1))

    def create_sequences(X_data, y_data, time_steps=1):
        Xs, ys = [], []
        for i in range(len(X_data) - time_steps):
            Xs.append(X_data[i:(i + time_steps)])
            ys.append(y_data[i + time_steps])
        return np.array(Xs), np.array(ys)

    for i, (train_index, test_index) in enumerate(tscv.split(X)):
        if len(train_index) <= TIMESTEPS: continue
        print(
            f"  Fold {i + 1}/{N_SPLITS}: Training on {len(train_index)} samples, testing on {len(test_index)} samples...")
        
        X_train_seq, y_train_seq = create_sequences(X_scaled[train_index], y_scaled[train_index], TIMESTEPS)
        
        # Convert to PyTorch tensors
        X_train_t = torch.from_numpy(X_train_seq).float()
        y_train_t = torch.from_numpy(y_train_seq).float()
        train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)

        # Build and train model
        model = model_class(input_dim=X_train_seq.shape[2], hidden_dim=50).to(device)
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        
        model.train()
        for epoch in range(15):
            for features, labels in train_loader:
                features, labels = features.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(features)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
        
        # Create test sequences and predict
        model.eval()
        with torch.no_grad():
            full_data_for_test = np.concatenate((X_scaled[train_index][-TIMESTEPS:], X_scaled[test_index]))
            X_test_seq, _ = create_sequences(full_data_for_test, np.zeros(len(full_data_for_test)), TIMESTEPS)
            X_test_t = torch.from_numpy(X_test_seq).float().to(device)
            y_pred_scaled = model(X_test_t).cpu().numpy()
            y_pred = y_scaler.inverse_transform(y_pred_scaled)
        
        y_test = y.iloc[test_index]
        all_metrics.append(calculate_metrics(y_test, y_pred.flatten(), is_solar))
        del model, X_train_t, y_train_t, train_loader;
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    return pd.DataFrame(all_metrics).mean().to_dict() if all_metrics else {}

## 6. Model Initialization & Logic Functions

In [8]:
# --- Sklearn-like Model Initializers ---
def get_xgboost_model():
    return xgb.XGBRegressor(tree_method='gpu_hist', objective='reg:squarederror', n_estimators=500,
                            learning_rate=0.05, max_depth=5, early_stopping_rounds=10, eval_metric='rmse')


def get_lightgbm_model():
    return lgb.LGBMRegressor(device='gpu', objective='regression', n_estimators=500,
                             learning_rate=0.05, num_leaves=31)


def get_randomforest_model():
    return RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42)


def get_svr_model():
    return SVR(kernel='linear', C=0.1)


# --- Univariate Model Logic ---
def prophet_logic(train_df, forecast_horizon):
    prophet_df = train_df.reset_index().rename(columns={'Timestamp': 'ds', 'BaseLoad': 'y'})
    model = Prophet()
    model.fit(prophet_df)
    future = model.make_future_dataframe(periods=forecast_horizon, freq='H')
    forecast = model.predict(future)
    return forecast['yhat'].values[-forecast_horizon:]


def ets_logic(train_df, forecast_horizon):
    model = ExponentialSmoothing(train_df['BaseLoad'], trend='add', seasonal='add', seasonal_periods=24).fit()
    return model.forecast(steps=forecast_horizon)


def sarima_logic(train_df, forecast_horizon):
    model = pm.auto_arima(train_df['BaseLoad'], seasonal=True, m=24, stepwise=True,
                          suppress_warnings=True, trace=False, error_action='ignore', n_jobs=-1)
    return model.predict(n_periods=forecast_horizon)


# --- PyTorch Deep Learning Models ---
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_layers=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, n_layers, batch_first=True)
        self.linear = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        return self.linear(lstm_out[:, -1, :])


class CNNLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, n_conv_filters=64, kernel_size=3):
        super(CNNLSTMModel, self).__init__()
        self.conv1d = nn.Conv1d(in_channels=input_dim, out_channels=n_conv_filters, kernel_size=kernel_size)
        self.relu = nn.ReLU()
        self.lstm = nn.LSTM(n_conv_filters, hidden_dim, batch_first=True)
        self.linear = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # Conv1d expects (batch, channels, seq_len)
        x = x.permute(0, 2, 1)
        conv_out = self.relu(self.conv1d(x))
        # LSTM expects (batch, seq_len, channels)
        conv_out = conv_out.permute(0, 2, 1)
        lstm_out, _ = self.lstm(conv_out)
        return self.linear(lstm_out[:, -1, :])


class TransformerModel(nn.Module):


    def __init__(self, input_dim, hidden_dim, n_heads=2, n_layers=2):
        super(TransformerModel, self).__init__()
        encoder_layers = nn.TransformerEncoderLayer(d_model=input_dim, nhead=n_heads, dim_feedforward=hidden_dim,
                                                    batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers=n_layers)
        self.linear = nn.Linear(input_dim, 1)

def forward(self, x):
    transformer_out = self.transformer_encoder(x)
    return self.linear(transformer_out[:, -1, :])

## 7. Main Execution Pipeline

In [9]:
    sklearn_models = {
    'LightGBM': get_lightgbm_model,
    'XGBoost': get_xgboost_model,
    'RandomForest': get_randomforest_model,
    # 'SVR': get_svr_model
}
univariate_models = {
    # 'Prophet': prophet_logic,
    # 'ETS': ets_logic,
}
if PMDARIMA_AVAILABLE:
    univariate_models['SARIMA'] = sarima_logic
pytorch_models = {
    'PyTorch_LSTM': LSTMModel,
    'PyTorch_CNN-LSTM': CNNLSTMModel,
    'PyTorch_Transformer': TransformerModel
}

segment_files = [f for f in os.listdir(SEGMENTS_DIR) if f.endswith('.csv')]
all_results = []

for segment_file in segment_files:
    print(f"\n{'=' * 60}")
    print(f"Processing Segment: {segment_file}")
    print(f"{'=' * 60}\n")
    
    df_full = pd.read_csv(os.path.join(SEGMENTS_DIR, segment_file), index_col='Timestamp', parse_dates=True)
    
    total_len = len(df_full)
    slice_len = int(total_len * 0.1)
    max_start = total_len - slice_len
    if max_start <= 0:
        print(f"Segment too small for sampling, using full data.")
        df = df_full.copy()
    else:
        random_start = np.random.randint(0, max_start)
        df = df_full.iloc[random_start: random_start + slice_len].copy()
        print(f"Using a 10% sample: {len(df)} rows from index {random_start} to {random_start + slice_len}.")

    df = reduce_mem_usage(df)
    is_solar = 'Solar' in segment_file
    segment_results = []

    # Run all model families
    for name, model_func in sklearn_models.items():
        try:
            segment_results.append({**run_sklearn_validation(df, model_func, is_solar), 'model': name})
        except Exception as e:
            print(f"ERROR running {name}: {e}")

    for name, model_logic in univariate_models.items():
        try:
            segment_results.append({**run_univariate_validation(df, name, model_logic, is_solar), 'model': name})
        except Exception as e:
            print(f"ERROR running {name}: {e}")
    
    for name, model_class in pytorch_models.items():
        try:
            dl_metrics = run_pytorch_validation(df, name, model_class, is_solar)
            if dl_metrics: segment_results.append({**dl_metrics, 'model': name})
        except Exception as e:
            print(f"ERROR running {name}: {e}")

    # Display Results
    if any(segment_results):
        results_df = pd.DataFrame([res for res in segment_results if res]).set_index('model').sort_index()
        primary_metric = 'WAPE' if is_solar else 'MAPE'
        if primary_metric not in results_df.columns: primary_metric = 'RMSE'  # Fallback
        cols_order = ['WAPE', 'sMAPE', 'MAPE', 'RMSE', 'MAE']
        results_df = results_df[[c for c in cols_order if c in results_df.columns]].sort_values(by=primary_metric)
        print(f"\n--- Model Comparison for: {segment_file} (on 10% sample) ---")
        print(results_df)
        results_df['segment'] = segment_file.replace('.csv', '')
        all_results.append(results_df.reset_index())
    
    del df, df_full;
    gc.collect()

print(f"\n{'=' * 60}")
print("PIPELINE EXECUTION COMPLETE")
print(f"{'=' * 60}\n")

if all_results:
    final_summary_df = pd.concat(all_results, ignore_index=True)


Processing Segment: Medium_Scale_Industries_Non-Solar.csv

Using a 10% sample: 96240 rows from index 547592 to 643832.
Memory usage of dataframe is 22.03 MB
Memory usage after optimization is: 11.11 MB
Decreased by 49.6%
--- Running validation for: get_lightgbm_model ---
  Fold 1/5: Training on 95400 samples, testing on 168 samples...
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 1656
[LightGBM] [Info] Number of data points in the train set: 95400, number of used features: 15
[LightGBM] [Info] Using GPU Device: NVIDIA GeForce RTX 4060 Laptop GPU, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 14 dense feature groups (1.46 MB) transferred to GPU in 0.003228 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 2760.423192
  Fold 2/5: Training on 95568 samples, testing on 168 samples...
[Li

## 8. Final Summary and Analysis

In [10]:
if all_results:
    print("--- Final Summary Across All Segments (from 10% sample) ---")
    display(final_summary_df)
else:
    print("No results were generated. Please check for errors in the pipeline.")

--- Final Summary Across All Segments (from 10% sample) ---


,model,WAPE,sMAPE,RMSE,MAE,segment
0,XGBoost,27.906544,14.570831,725.524792,563.586829,Medium_Scale_Industries_Non-Solar
1,LightGBM,28.071309,14.673375,727.549490,566.462756,Medium_Scale_Industries_Non-Solar
2,PyTorch_CNN-LSTM,29.636612,15.480753,762.136287,597.523730,Medium_Scale_Industries_Non-Solar
3,RandomForest,29.843738,15.465255,763.062351,600.723680,Medium_Scale_Industries_Non-Solar
4,PyTorch_LSTM,30.176004,15.659470,769.103907,606.400024,Medium_Scale_Industries_Non-Solar
5,PyTorch_Transformer,30.781525,16.010349,781.729901,620.832996,Medium_Scale_Industries_Non-Solar
6,XGBoost,34.453289,18.332668,995.727757,711.036615,Medium_Scale_Industries_Solar
7,LightGBM,34.486090,18.363061,1000.616553,711.337288,Medium_Scale_Industries_Solar
8,RandomForest,35.718125,18.605078,1045.339689,735.675587,Medium_Scale_Industries_Solar
9,PyTorch_Transformer,39.282654,20.553289,1138.270440,801.083093,Medium_Scale_Industries_Solar


In [11]:
final_summary_df.to_csv('model_comparison_summary.csv', index=False)